# 05 — Evaluation, comparison, and report tables

This notebook aggregates all completed runs, plots the comparison, and creates per-label result tables for the best model.

In [ ]:

from pathlib import Path
import sys, os, json, time

# Locate the project root whether this folder is in /kaggle/working or uploaded as a Kaggle dataset.
candidates = [Path.cwd(), Path('/kaggle/working/ladi_v2_gcn_project')]
if Path('/kaggle/input').exists():
    candidates += list(Path('/kaggle/input').glob('*'))
PROJECT_ROOT = None
for c in candidates:
    if (c / 'src' / 'ladi_config.py').exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find project src/. Upload/unzip the full ladi_v2_gcn_project folder first.')
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ladi_config import LABEL_COLS_V2A
from src.ladi_train import collect_run_metrics
from src.ladi_metrics import per_label_table

OUT_DIR = Path('/kaggle/working/ladi_v2_outputs')
RUN_DIR = OUT_DIR / 'runs'
EVAL_DIR = OUT_DIR / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
summary = collect_run_metrics(RUN_DIR)
summary.to_csv(EVAL_DIR / 'model_comparison_summary.csv', index=False)
display(summary)


In [ ]:
if not summary.empty:
    plot_df = summary.sort_values('test_macro_ap', ascending=True)
    plt.figure(figsize=(8, max(4, 0.45 * len(plot_df))))
    plt.barh(plot_df['run_name'], plot_df['test_macro_ap'])
    plt.xlabel('Test macro average precision')
    plt.title('LADI-v2 model comparison')
    plt.tight_layout()
    plt.savefig(EVAL_DIR / 'model_comparison_macro_ap.png', dpi=180)
    plt.show()


In [ ]:
# Per-label table for the best model by test macro AP.
if not summary.empty:
    best_run = summary.iloc[0]['run_name']
    pred_path = RUN_DIR / best_run / 'test_predictions.npz'
    arr = np.load(pred_path)
    table = per_label_table(arr['y_true'], arr['y_prob'], LABEL_COLS_V2A, thresholds=arr['thresholds'])
    table = table.sort_values('AP', ascending=False)
    table.to_csv(EVAL_DIR / f'{best_run}_per_label_test.csv', index=False)
    print('Best run:', best_run)
    display(table)


In [ ]:
# Write a compact markdown summary you can paste into the dissertation/project report.
if not summary.empty:
    md = []
    md.append('# LADI-v2 experiment summary\n')
    md.append(summary.to_markdown(index=False))
    md.append('\n\nPrimary selection metric: validation macro AP. Test macro AP is reported for final comparison. Threshold-tuned F1 uses validation-set thresholds only.\n')
    (EVAL_DIR / 'experiment_summary.md').write_text('\n'.join(md))
    print((EVAL_DIR / 'experiment_summary.md').read_text()[:1500])
